In [21]:
from sklearn.preprocessing import StandardScaler
from dataclasses import dataclass, field
from sklearn.linear_model import Ridge
from datetime import datetime
from fredapi import Fred
import yfinance as yf
import pandas as pd
import cvxpy as cp
import numpy as np
import os

In [12]:
class DataStore:
  def __init__(self, debug: bool = False):
    self.debug = debug

  def get_data(self, tickers: list, start: str, end: str, fred_api_key: str,
               interval: str = "1d", benchmark: str = "^GSPC",
               cyclical: str = "XLY", defensive: str = "XLP"):
    tmp = []
    for t in tickers:
      if isinstance(t, list):
        tmp.extend(t)
      elif isinstance(t, str):
        tmp.append(t)
      else:
        print(f"Warning: Skipping {t} | type: {type(t)}")
    tickers_clean = sorted(set(tmp))

    df_path = f"prices_{start}_{end}.parquet"
    if os.path.exists(df_path):
      df = pd.read_parquet(df_path)
    else:
      df = yf.download(tickers_clean, start=start, end=end, interval=interval)["Close"]
      df.to_parquet(df_path)

    returns = df.ffill().pct_change(fill_method=None)

    factors = self._get_factors(start, end, fred_api_key, benchmark, cyclical, defensive)
    return returns, factors

  def _fetch_equity_factors(self, start, end, benchmark="^GSPC",
                            cyclical="XLY", defensive="XLP"):
    r = yf.download([benchmark, "^IRX", cyclical, defensive],
                    start=start, end=end)["Close"]
    rets = r[[benchmark, cyclical, defensive]].pct_change(fill_method=None)
    rf_daily = (r["^IRX"].ffill() / 100) / 252

    factors = pd.DataFrame({
      "Benchmark": rets[benchmark] - rf_daily,
      "Spread": rets[cyclical] - rets[defensive],
    })
    return factors

  def _get_breakeven_inf(self, fred) -> pd.Series:
    return fred.get_series("DGS5") - fred.get_series("DFII5")

  def _fetch_fred(self, fred_api_key) -> pd.DataFrame:
    fred = Fred(api_key=fred_api_key)
    cols = {
      "NFCI":         fred.get_series("NFCI"),
      "USD":          fred.get_series("DTWEXBGS"),
      "TIPS10Y":      fred.get_series("DFII10"),
      "HighYieldOAS": fred.get_series("BAMLH0A0HYM2"),
      "BreakevenInf": self._get_breakeven_inf(fred),
    }
    out = pd.concat(cols, axis=1).sort_index()
    out.index = pd.to_datetime(out.index)
    return out

  def _get_factors(self, start, end, fred_api_key, benchmark="^GSPC",
                   cyclical="XLY", defensive="XLP") -> pd.DataFrame:
    path = f"factors_{start}_{end}.parquet"
    if os.path.exists(path):
      return pd.read_parquet(path)

    fred_df = self._fetch_fred(fred_api_key)
    eq_df = self._fetch_equity_factors(start, end, benchmark, cyclical, defensive)

    combined = pd.concat([fred_df, eq_df], axis=1, join="outer").sort_index()
    combined = combined.loc[start:end]
    combined.to_parquet(path)

    if self.debug:
      print("First valid date per factor:")
      print(combined.apply(pd.Series.first_valid_index))
    return combined

In [15]:
@dataclass
class BootstrapParams:
  n_bootstraps: int = 1000
  block_length: int = 10
  horizon: int = None
  circular: bool = False
  seed: int = None

  def __post_init__(self):
    if self.n_bootstraps < 1:
      raise ValueError(f"Invalid n_bootstraps: {self.n_bootstraps}")

    if self.block_length < 1:
      raise ValueError(f"Invalid block_length: {self.block_length}")

    if self.horizon is not None and self.horizon < 1:
      raise ValueError(f"Invalid horizon: {self.horizon}")

In [16]:
class BlockBootstrap:
  def __init__(self, params: BootstrapParams = None, **kwargs):
    self.p = params or BootstrapParams()
    self.rng = np.random.default_rng(self.fp.seed)
    super().__init__(**kwargs)

  def _validate(self, returns: pd.DataFrame) -> int:
    if returns.isna().any().any():
      raise ValueError("returns contains NaNs; clean/align them first (e.g. dropna()).")

    T = len(returns)
    if self.fp.block_length > T:
      raise ValueError(f"block_length ({self.fp.block_length}) exceeds sample length ({T})")

    return self.fp.horizon if self.fp.horizon is not None else T

  def _sample_indices(self, T: int, horizon: int) -> np.ndarray:
    L = self.fp.block_length
    n_blocks = int(np.ceil(horizon / L))

    n_starts = T if self.fp.circular else T - L + 1
    starts = self.rng.integers(0, n_starts, size=(self.p.n_bootstraps, n_blocks))

    idx = (starts[:, :, None] + np.arange(L)).reshape(self.p.n_bootstraps, -1)
    idx = idx[:, :horizon]

    return idx % T if self.fp.circular else idx

  def resample(self, returns: pd.DataFrame, as_dataframe: bool = False):
    horizon = self._validate(returns)
    values = returns.to_numpy()

    idx = self._sample_indices(len(values), horizon)
    samples = values[idx]

    if not as_dataframe:
      return samples

    index = pd.MultiIndex.from_product(
      [range(self.p.n_bootstraps), range(horizon)], names=["bootstrap", "step"]
    )
    return pd.DataFrame(
      samples.reshape(-1, samples.shape[2]), index=index, columns=returns.columns
    )

In [17]:
FACTOR_KINDS = {
  "Benchmark":    "compound",
  "Spread":       "compound",
  "USD":          "pct",
  "NFCI":         "diff",
  "BreakevenInf": "diff",
  "HighYieldOAS": "diff",
  "TIPS10Y":      "diff",
}
@dataclass
class FilterParams:
  min_length: tuple = (5, "y")
  debug: bool = True
  factor_kinds: dict = field(default_factory=lambda: dict(FACTOR_KINDS))
  week_anchor: str = "W-FRI"
  min_days_per_week: int = 3
  min_weeks: int = 26
  window_size: int = 104
  ridge_alpha: float = 1.0
  beta_change_window:int = 10

  def __post_init__(self):
    if not self.min_length[1] in ["d", "m", "y"]:
      raise ValueError(f"Invalid time period: {self.min_length[1]}")

    if self.min_length[0] <= 0:
      raise ValueError(f"Invalid min length: {self.min_length[0]}")

    if not 1 <= self.min_days_per_week <= 7:
      raise ValueError(f"Invalid min_days_per_week: {self.min_days_per_week}")

    if self.min_weeks <= 0:
      raise ValueError(f"Invalid min_weeks: {self.min_weeks}")

In [27]:
class Optimizer:
  def __init__(self, filter_params, **kwargs):
    self.fp = filter_params
    super().__init__(**kwargs)

  def _select_assets(self, unit, quality, n_select):
    N_assets, K_factors = unit.shape
    I = np.identity(K_factors)
    z = cp.Variable(N_assets)

    A = I + self.fp.factod_dir_importance * quality * cp.quad_form(unit, z)
    obj = cp.log_det(A)

    constraints = [
        z >= 0,
        z <= 1,
        cp.sum(z) == n_select
    ]

    problem = cp.Problem(obj, constraints)

    problem.solve(solver=cp.CLARABEL)

    if problem.status not in (
        cp.OPTIMAL,
        cp.OPTIMAL_INACCURATE,
    ):
        raise RuntimeError(problem.status)

    return z.value, problem.value, A.value




In [19]:
class Filter(BlockBootstrap, LogDetObjectiveOptimizer):
  def __init__(
    self,
    filter_params: FilterParams,
    bootstrap_params: BootstrapParams
  ):
    super().__init__(bootstrap_params)

    self.fp = filter_params
    self.bootstrap_params = bootstrap_params

  @staticmethod
  def _naive(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.index = pd.to_datetime(df.index).tz_localize(None)
    return df.sort_index()

  def _compound_weekly(self, rets: pd.DataFrame) -> pd.DataFrame:
    g = np.log1p(rets).resample(self.fp.week_anchor)
    weekly = np.expm1(g.sum(min_count=1))
    return weekly.where(rets.resample(self.fp.week_anchor).count() >= self.fp.min_days_per_week)

  def _to_weekly_returns(self, returns: pd.DataFrame) -> pd.DataFrame:
    return self._compound_weekly(self._naive(returns))

  def _to_weekly_factors(self, factors: pd.DataFrame) -> pd.DataFrame:
    factors = self._naive(factors)
    kinds = self.fp.factor_kinds

    unknown = set(factors.columns) - set(kinds)
    if unknown:
      raise ValueError(f"No aggregation rule for factors: {sorted(unknown)}")

    comp = [c for c in factors.columns if kinds[c] == "compound"]
    lvl = [c for c in factors.columns if kinds[c] != "compound"]

    parts = []
    if comp:
      parts.append(self._compound_weekly(factors[comp]))
    if lvl:
      wk = factors[lvl].resample(self.fp.week_anchor).last().ffill(limit=1)
      for c in lvl:
        wk[c] = wk[c].diff() if kinds[c] == "diff" else wk[c].pct_change(fill_method=None)
      parts.append(wk)

    return pd.concat(parts, axis=1)[factors.columns]

  def _time_horizon_filter(self, returns: pd.DataFrame) -> pd.DataFrame:
    end = pd.to_datetime(returns.index.max())
    min_l, tp = self.fp.min_length

    min_start = end - pd.DateOffset(
      years=min_l if tp == "y" else 0,
      months=min_l if tp == "m" else 0,
      days=min_l if tp == "d" else 0,
    )
    if self.fp.debug:
      print(f"Min start: {min_start}")

    drop = [t for t in returns.columns
            if pd.to_datetime(returns[t].dropna().index.min()) > min_start]
    return returns.drop(columns=drop).dropna()

  def _get_bootstraps(self, returns: pd.DataFrame, factors: pd.DataFrame):
    common = returns.dropna().index.intersection(factors.dropna().index)
    Y_df = returns.loc[common]
    scaler = StandardScaler()
    X_df = factors.loc[common]

    assets, fnames = Y_df.columns, X_df.columns
    N, K = len(assets), len(fnames)

    draws = self.resample(pd.concat([Y_df, X_df], axis=1))
    B = draws.shape[0]
    Yb, Xb = draws[:, :, :N], draws[:, :, N:]

    coefs = np.empty((B, N, K))
    for b in range(B):
      scaler = StandardScaler()
      Xb[b] = scaler.fit_transform(Xb[b])
      coefs[b] = self._fit_ridge(X=Xb[b], y=Yb[b]).coef_/ scaler.scale_

    lower = np.percentile(coefs, 2.5, axis=0)
    upper = np.percentile(coefs, 97.5, axis=0)

    p_pos = (coefs > 0).mean(axis=0)
    stability = np.maximum(p_pos, 1 - p_pos)

    factor_index = pd.Index(fnames, name="Factor")

    percentile_intervals = pd.DataFrame(
      np.stack([lower, upper], axis=2).transpose(1, 0, 2).reshape(K, N * 2),
      index=factor_index,
      columns=pd.MultiIndex.from_product(
        [assets, ["Lower CI (2.5%)", "Upper CI (97.5%)"]],
        names=["Asset", "Bound"],
      ),
    )

    sign_stability = pd.DataFrame(
      stability.T,
      index=factor_index,
      columns=pd.Index(assets, name="Asset"),
    )

    beta_estimation_var = np.var(
      coefs,
      axis=0,
      ddof=1
    )

    return percentile_intervals, sign_stability, p_pos, beta_estimation_var, coefs


  def _fit_ridge(self, X, y):
    return Ridge(
      alpha=self.fp.ridge_alpha,
      fit_intercept=True
    ).fit(X, y)

  def _rolling_factor_params(
    self,
    returns: pd.DataFrame,
    factors: pd.DataFrame
  ):
    idx = returns.index.intersection(factors.index).sort_values()
    returns = returns.loc[idx]
    factors = factors.loc[idx]

    if returns.isna().any().any() or factors.isna().any().any():
      raise ValueError("Missing data in regression inputs")

    X = factors.to_numpy(dtype=float)
    Y = returns.to_numpy(dtype=float)

    T, N = Y.shape
    K = X.shape[1]
    w = self.fp.window_size

    if T <= w:
      raise ValueError(
        f"Need more than {w} weekly observations; got {T}"
      )

    betas_raw = np.full((T, N, K), np.nan)
    betas_std = np.full((T, N, K), np.nan)
    delta_r2 = np.full((T, N, K), np.nan)

    r2_in = np.full((T, N), np.nan)
    r2_oos = np.full((T, N), np.nan)

    rmse_oos = np.full((T, N), np.nan)
    resid_vol_in = np.full((T, N), np.nan)
    oos_resid = np.full((T, N), np.nan)

    cumulative_sse = np.zeros(N)
    cumulative_sst = np.zeros(N)
    cumulative_sse_wo = np.zeros((K, N))

    def safe_r2(sse, sst):
      ratio = np.full_like(sse, np.nan, dtype=float)
      np.divide(sse, sst, out=ratio, where=sst > 0)
      return 1.0 - ratio

    def fit_window(start, stop):
      x_train = X[start:stop]
      y_train = Y[start:stop]

      scaler = StandardScaler()
      x_scaled = scaler.fit_transform(x_train)

      model = self._fit_ridge(x_scaled, y_train)

      train_resid = y_train - model.predict(x_scaled)
      y_mean = y_train.mean(axis=0)

      train_sse = np.sum(train_resid**2, axis=0)
      train_sst = np.sum((y_train - y_mean)**2, axis=0)

      return (
        model,
        scaler,
        safe_r2(train_sse, train_sst),
        train_resid.std(axis=0, ddof=0),
        y_mean
      )

    for t in range(w, T):
      model, scaler, train_r2, train_vol, y_mean = (
        fit_window(t - w, t)
      )
      betas_std[t - 1] = model.coef_
      betas_raw[t - 1] = model.coef_ / scaler.scale_

      r2_in[t - 1] = train_r2
      resid_vol_in[t - 1] = train_vol

      x_test = scaler.transform(X[t:t + 1])
      prediction = model.predict(x_test)[0]

      error = Y[t] - prediction
      oos_resid[t] = error

      cumulative_sse += error**2
      cumulative_sst += (Y[t] - y_mean)**2

      x_win = scaler.transform(X[t - w : t])
      y_win = Y[t - w : t]
      for k in range(K):
        if K > 1:
          m_k = self._fit_ridge(np.delete(x_win, k, axis=1), y_win)
          pred_k = m_k.predict(np.delete(x_test, k, axis=1))[0]
        else:
          pred_k = y_mean
        cumulative_sse_wo[k] += (Y[t] - pred_k)**2

      n_oos = t - w + 1

      r2_oos[t] = safe_r2(cumulative_sse, cumulative_sst)
      rmse_oos[t] = np.sqrt(cumulative_sse / n_oos)

      delta = np.full((K, N), np.nan)
      np.divide(
        cumulative_sse_wo - cumulative_sse,
        cumulative_sst,
        out=delta,
        where=cumulative_sst > 0
      )
      delta_r2[t] = delta.T

    model, scaler, train_r2, train_vol, _ = (
      fit_window(T - w, T)
    )

    betas_std[-1] = model.coef_
    betas_raw[-1] = model.coef_ / scaler.scale_
    r2_in[-1] = train_r2
    resid_vol_in[-1] = train_vol

    residuals_df = pd.DataFrame(
      oos_resid,
      index=idx,
      columns=returns.columns
    )

    resid_vol_oos = (
      residuals_df
      .expanding(min_periods=2)
      .std(ddof=0)
      .to_numpy()
    )

    beta_columns = pd.MultiIndex.from_product(
      [returns.columns, factors.columns],
      names=["Asset", "Factor"]
    )

    def make_beta_df(values):
      return pd.DataFrame(
        values.reshape(T, N * K),
        index=idx,
        columns=beta_columns
      )

    metrics = [
      "R2_in",
      "R2_oos",
      "RMSE_oos",
      "ResidVol_in",
      "ResidVol_oos"
    ]

    values = np.stack(
      [
        r2_in,
        r2_oos,
        rmse_oos,
        resid_vol_in,
        resid_vol_oos
      ],
      axis=2
    )

    stats_df = pd.DataFrame(
      values.reshape(T, N * len(metrics)),
      index=idx,
      columns=pd.MultiIndex.from_product(
        [returns.columns, metrics],
        names=["Asset", "Metric"]
      )
    )

    return (
      make_beta_df(betas_raw),
      make_beta_df(betas_std),
      make_beta_df(delta_r2),
      stats_df,
      residuals_df
    )


  def _shrink_vasicek(betas_raw: pd.DataFrame, est_var: pd.DataFrame):
    fac = betas_raw.columns.get_level_values("Factor").tolist()
    bcast = lambda d: d[fac].set_axis(betas_raw.columns, axis=1)

    grp = lambda d: d.T.groupby(level="Factor")
    prior_mean = grp(betas_raw).mean().T
    obs_var = grp(betas_raw).var().T
    noise = grp(est_var).mean().T
    prior_var = (obs_var - noise).clip(lower=0)

    pv, pm = bcast(prior_var), bcast(prior_mean)
    w_raw = (pv / (pv + est_var)).fillna(0.0)
    return w_raw * betas_raw + (1 - w_raw) * pm, w_raw




  def _get_factor_exposure_features(
    self,
    returns: pd.DataFrame,
    factors: pd.DataFrame
  ) -> tuple[pd.DataFrame, pd.DataFrame]:
    weekly_returns = self._to_weekly_returns(returns).dropna()
    weekly_factors = self._to_weekly_factors(factors).dropna()

    common = weekly_returns.index.intersection(weekly_factors.index)
    X = weekly_factors.loc[common]
    Y = weekly_returns.loc[common]

    w = self.fp.window_size
    percentile_intervals, sign_stability, p_pos, beta_estimation_var, coefs = (
      self._get_bootstraps(
        Y.iloc[-w:],
        X.iloc[-w:]
      )
    )

    betas_raw, betas_std, stats_df, residuals = (
      self._rolling_factor_params(Y, X)
    )

    factor_fields = pd.concat(
      [percentile_intervals.stack("Asset"),
      sign_stability.stack().rename("SignStability")],
      axis=1,
    )
    factor_part = pd.concat(
      {f: factor_fields.xs(f, level="Factor") for f in X.columns}, axis=1
    )
    factor_part.columns.names = ["Factor", "Field"]

    model_cols = ["R2_in", "R2_oos", "RMSE_oos", "ResidVol"]
    model_part = pd.concat(
      {"Model": stats_df.iloc[-1].unstack("Metric")[model_cols]}, axis=1
    )
    model_part.columns.names = ["Factor", "Field"]

    summary = pd.concat([factor_part, model_part], axis=1)

    betas_adjusted = self._shrink_vasicek(betas_raw, beta_estimation_var)

    beta_change = betas_adjusted.diff(n=self.fp.beta_change_window)

    return betas_adjusted, beta_change, summary, coefs


  def _risk_exposure_similarity(self, betas, factors: pd.DataFrame):
    if isinstance(betas, pd.Series):
      assets = betas.index.get_level_values("Asset").unique()
      betas = betas.unstack("Factor").reindex(assets)

    missing = [c for c in factors.columns if c not in betas.columns]
    if missing:
      raise ValueError(f"No betas for factors: {missing}")
    betas = betas[factors.columns]

    sigma = factors.dropna().cov().to_numpy()
    eigval, eigvec = np.linalg.eigh(sigma)
    sqrt_sigma = (eigvec * np.sqrt(np.clip(eigval, 0.0, None))) @ eigvec.T

    B = betas.to_numpy(dtype=float) @ sqrt_sigma

    norms = np.linalg.norm(B, axis=1, keepdims=True)
    B_hat = np.divide(B, norms, out=np.full_like(B, np.nan), where=norms > 0)

    cos = np.clip(B_hat @ B_hat.T, -1.0, 1.0)

    assets = betas.index
    exposures = pd.DataFrame(B, index=assets, columns=factors.columns)
    unit = pd.DataFrame(B_hat, index=assets, columns=factors.columns)
    cos_sim = pd.DataFrame(
      cos,
      index=pd.Index(assets, name="Asset"),
      columns=pd.Index(assets, name="Asset"),
    )

    return exposures, unit, cos_sim

  def _get_ref_redundancy(
    self,
    unit: pd.DataFrame,
    A_prev: np.ndarray,
    quality: np.ndarray,
    selected_mask: np.ndarray
  ) -> float:
    U_matrix = unit.to_numpy(dtype=float)
    unselected_indices = np.where(~selected_mask)[0]

    if len(unselected_indices) == 0:
      return 0.0

    U_unselected = U_matrix[unselected_indices]
    q_unselected = quality[unselected_indices] if quality is not None else 1.0
    kappa = getattr(self.fp, 'kappa', 1.0)

    try:
      A_inv = np.linalg.inv(A_prev)
    except np.linalg.LinAlgError:
      A_inv = np.linalg.pinv(A_prev)

    quad_forms = np.sum((U_unselected @ A_inv) * U_unselected, axis=1)

    vals = 1.0 + kappa * q_unselected * quad_forms
    vals = np.maximum(vals, 1e-12)

    redundancy = float(np.max(np.log(vals)))

    return redundancy

  def _get_uncertainty(
      self,
      redundancy: pd.Series | float | np.ndarray,
      coefs: np.ndarray,
      unit: pd.DataFrame,
      quality: np.ndarray,
      m: int,
      factor_cov: pd.DataFrame | np.ndarray,
  ) -> float:

    B, N, K = coefs.shape

    if B < 2:
        raise ValueError("At least 2 bootstrap samples required")

    if not 1 <= m <= N:
        raise ValueError("m must be between 1 and N")

    quality = np.asarray(quality, dtype=float)

    if quality.shape != (N,) or unit.shape != (N, K):
        raise ValueError("Incompatible asset or factor dimensions")

    if isinstance(factor_cov, pd.DataFrame):
        sigma = factor_cov.loc[
            unit.columns, unit.columns
        ].to_numpy(dtype=float)
    else:
        sigma = np.asarray(factor_cov, dtype=float)

    if sigma.shape != (K, K):
        raise ValueError("Invalid factor covariance shape")

    eigval, eigvec = np.linalg.eigh(
        (sigma + sigma.T) / 2
    )

    sqrt_sigma = (
        eigvec * np.sqrt(np.clip(eigval, 0.0, None))
    ) @ eigvec.T
    exposures = coefs @ sqrt_sigma

    norms = np.linalg.norm(
        exposures, axis=2, keepdims=True
    )

    if np.any(norms <= 1e-12):
        raise ValueError("Zero bootstrap factor exposure")

    normalized = exposures / norms

    ref = np.asarray(redundancy, dtype=float)

    if ref.ndim > 1 or (
        ref.ndim == 1 and ref.shape != (B,)
    ):
        raise ValueError(
            "redundancy must be scalar or length B"
        )

    excess_coverage = np.empty(B, dtype=float)

    for b in range(B):
      unit_b = pd.DataFrame(
          normalized[b],
          index=unit.index,
          columns=unit.columns,
      )

      _, C_m, _, _ = self._select_assets(
          unit_b, quality, m
      )

      if m == 1:
          C_prev = 0.0
      else:
        _, C_prev, _, _ = self._select_assets(
          unit_b, quality, m - 1
        )

      delta_C = C_m - C_prev

      excess_coverage[b] = delta_C - (
        ref[b] if ref.ndim == 1 else ref.item()
      )

    return float(np.std(excess_coverage, ddof=1))

  def _select_opt_n_assets(self, unit: pd.DataFrame, quality: np.ndarray, coefs:np.ndarray, min_assets: int = 2):
    n_assets = unit.shape[0]
    C_m_prev = 0.0
    opt_selection = np.zeros(n_assets, dtype=bool)
    A_prev = np.eye(unit.shape[1])

    for m in range(min_assets, n_assets):
      selection_w, C_m, selected_mask, A_m = self._select_assets(unit, quality, m)
      delta_C_observed = C_m - C_m_prev

      redundancy_ref = self._get_ref_redundancy(unit, A_prev, quality, selected_mask)
      excess_coverage_uncertainty = self._get_uncertainty(redundancy_ref)
      excess_coverage = delta_C_observed - redundancy_ref

      if excess_coverage < excess_coverage_uncertainty:
        break

      A_prev = A_m
      C_m_prev = C_m
      opt_selection = selection_w

    return opt_selection

  def select_assets(self, returns, factors, quality):
    beta_adjusted, beta_change, summary, coefs = self._get_factor_exposure_features()

    exposures, unit, cos_sim = self.risk_exposure_similarity(beta_adjusted, factors)

    selection_vector = self._select_opt_n_assets(unit, quality, coefs)
